# 7-MLET Datathon — Política Comercial Sintética, RAG/Suitability e Golden Set

**Fala rapaziada!**

Esse notebook cobre **exatamente** a parte que o notebook de bandits (`7mlet_bandits_step_by_step`) **não** entrega: o item de governança/LLM do enunciado. Aqui vocês têm um esqueleto runnable de:

1. **Documentos sintéticos de política comercial** — o corpus fictício de diretrizes da instituição (`data/policies/`).
2. **Camada de suitability + RAG** — recupera a política relevante e checa se a oferta é adequada ao perfil, com *reason codes*.
3. **Golden set** — `data/golden_set/evaluation_cases.jsonl` com 20+ casos (típicos, borda, segmento, adversariais), cada um com pass/fail explícito.
4. **Avaliação offline reproduzível** — roda o golden set contra o motor de suitability e reporta métricas.

Onde isso encaixa no enunciado:
- **Etapa 2** — documentos de política comercial e suitability fazem parte do enriquecimento sintético.
- **Etapa 4** — golden set (mínimo 20 casos) + avaliação offline reproduzível.
- **Etapa 5** — log auditável de decisão com *reason codes*.
- **Etapa 8** — narrativa de governança (System Card / guardrails).

> **Importante:** tudo aqui é **sintético e fictício**. Nenhuma regra comercial real, nenhum dado de cliente real. É isso que o enunciado exige.


## 0) Setup e estrutura de pastas

Só dependências leves: `scikit-learn` (para o retriever TF-IDF) e a stdlib. Nada de chave de API. Mais adiante eu mostro como trocar o retriever TF-IDF por ChromaDB + um LLM local (Qwen) se vocês quiserem subir o nível.


In [1]:
from __future__ import annotations

import json
import textwrap
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any

import numpy as np

# Pastas de saida (tudo local, descartavel)
BASE_DIR = Path.cwd() / "datathon_item9"
POLICIES_DIR = BASE_DIR / "data" / "policies"
GOLDEN_DIR = BASE_DIR / "data" / "golden_set"
REPORTS_DIR = BASE_DIR / "reports"

for folder in (POLICIES_DIR, GOLDEN_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

RNG = np.random.default_rng(7)

print("Base dir   :", BASE_DIR)
print("Policies   :", POLICIES_DIR)
print("Golden set :", GOLDEN_DIR)
print("Reports    :", REPORTS_DIR)


Base dir   : c:\Users\ricar\Github\i\tmp\7mlet\datathon_item9
Policies   : c:\Users\ricar\Github\i\tmp\7mlet\datathon_item9\data\policies
Golden set : c:\Users\ricar\Github\i\tmp\7mlet\datathon_item9\data\golden_set
Reports    : c:\Users\ricar\Github\i\tmp\7mlet\datathon_item9\reports


## A) Catálogo de ofertas + regras estruturadas de suitability

A ideia central: as regras de suitability nascem **estruturadas** (dataclass), e a partir delas eu gero os **documentos de política em texto** que o RAG vai recuperar. Assim a checagem é **determinística** (bom pro golden set ter pass/fail estável) e o texto é **humano** (bom pro assistente LLM explicar).

Cada braço/oferta carrega: nível de risco, saldo mínimo exigido, se exige reserva de emergência, faixa etária permitida e desconto máximo. São atributos **fictícios** — vocês ajustam pro problema de vocês.


In [2]:
@dataclass(frozen=True)
class OfferPolicy:
    arm: int
    offer_name: str
    risk_level: str            # "baixo" | "medio" | "alto"
    min_balance: float         # saldo minimo exigido
    requires_reserve: bool     # exige reserva de emergencia (prior contact como proxy)
    min_age: int
    max_age: int
    max_discount: float        # teto de desconto permitido (fracao)
    rationale: str             # justificativa de negocio (sintetica)


# Catalogo FICTICIO de ofertas para o problema de marketing bancario.
OFFER_CATALOG: list[OfferPolicy] = [
    OfferPolicy(0, "Sem Incentivo", "baixo", 0.0, False, 18, 99, 0.00,
                "Controle: contato sem oferta. Sempre adequado, nunca viola suitability."),
    OfferPolicy(1, "Desconto de Tarifa", "baixo", 500.0, False, 18, 99, 0.15,
                "Oferta de baixo risco; exige saldo minimo modesto para evitar abrir conta apenas pela tarifa."),
    OfferPolicy(2, "Cashback", "medio", 1500.0, False, 21, 75, 0.20,
                "Risco medio; cashback so e sustentavel para clientes com saldo recorrente."),
    OfferPolicy(3, "Pacote Premium", "alto", 5000.0, True, 25, 65, 0.25,
                "Produto de risco alto; exige reserva de emergencia e faixa etaria de renda ativa."),
]

OFFERS_BY_ARM = {offer.arm: offer for offer in OFFER_CATALOG}

import pandas as pd
catalog_df = pd.DataFrame([asdict(o) for o in OFFER_CATALOG])
catalog_df


,arm,offer_name,risk_level,min_balance,requires_reserve,min_age,max_age,max_discount,rationale
0,0,Sem Incentivo,baixo,0.0,False,18,99,0.00,"Controle: contato sem oferta. Sempre adequado,..."
1,1,Desconto de Tarifa,baixo,500.0,False,18,99,0.15,Oferta de baixo risco; exige saldo minimo mode...
2,2,Cashback,medio,1500.0,False,21,75,0.20,Risco medio; cashback so e sustentavel para cl...
3,3,Pacote Premium,alto,5000.0,True,25,65,0.25,Produto de risco alto; exige reserva de emerge...


In [3]:
def render_policy_document(offer: OfferPolicy) -> str:
    reserve_line = (
        "- Exige reserva de emergencia comprovada (proxy: contato/relacionamento previo)."
        if offer.requires_reserve
        else "- Nao exige reserva de emergencia previa."
    )
    return textwrap.dedent(f"""\
    # Politica Comercial Sintetica — {offer.offer_name} (arm {offer.arm})

    > Documento FICTICIO para fins academicos (Datathon 7-MLET). Nao representa
    > regra comercial real de nenhuma instituicao.

    ## Classificacao
    - Nivel de risco do produto: **{offer.risk_level}**.
    - Justificativa de negocio: {offer.rationale}

    ## Criterios de suitability (adequacao ao perfil)
    - Saldo minimo exigido: R$ {offer.min_balance:,.2f}.
    {reserve_line}
    - Faixa etaria permitida: {offer.min_age} a {offer.max_age} anos.
    - Desconto maximo autorizado: {offer.max_discount:.0%}.

    ## Guardrails
    - Ofertar este produto a cliente fora da faixa etaria permitida VIOLA suitability.
    - Ofertar este produto a cliente abaixo do saldo minimo VIOLA suitability.
    - Conceder desconto acima do teto autorizado VIOLA a politica de pricing.
    - Decisoes sensiveis exigem humano no loop e log auditavel com reason codes.
    """)


policy_paths: list[Path] = []
for offer in OFFER_CATALOG:
    path = POLICIES_DIR / f"policy_arm_{offer.arm}_{offer.offer_name.lower().replace(' ', '_')}.md"
    path.write_text(render_policy_document(offer), encoding="utf-8")
    policy_paths.append(path)

# Documento transversal de suitability + LGPD (sintetico)
general_path = POLICIES_DIR / "policy_00_geral_suitability_lgpd.md"
general_path.write_text(textwrap.dedent("""\
    # Politica Geral de Suitability e Privacidade (Sintetica)

    > Documento FICTICIO para fins academicos (Datathon 7-MLET).

    ## Principios
    - Toda oferta deve ser adequada ao perfil de risco e a capacidade financeira do cliente.
    - Em caso de duvida de adequacao, a politica de controle (Sem Incentivo) deve prevalecer.
    - Nenhuma decisao automatizada de alto risco e tomada sem humano no loop.

    ## Privacidade (LGPD)
    - Nao usar dados reais de clientes, identificadores, renda, genero ou raca.
    - Minimizacao: usar apenas atributos necessarios a decisao de oferta.
    - Toda decisao e registrada com reason codes para auditoria e retencao limitada.
    """), encoding="utf-8")
policy_paths.append(general_path)

print(f"{len(policy_paths)} documentos de politica gerados em: {POLICIES_DIR}")
for path in policy_paths:
    print(" -", path.name)


5 documentos de politica gerados em: c:\Users\ricar\Github\i\tmp\7mlet\datathon_item9\data\policies
 - policy_arm_0_sem_incentivo.md
 - policy_arm_1_desconto_de_tarifa.md
 - policy_arm_2_cashback.md
 - policy_arm_3_pacote_premium.md
 - policy_00_geral_suitability_lgpd.md


## B) Camada de suitability + RAG

Duas peças que trabalham juntas:

- **Motor de suitability (determinístico)** — recebe contexto do cliente + braço escolhido e devolve um veredito `aprovado / reprovado` com **reason codes**. É o guardrail "duro". Determinístico de propósito: é o que dá pass/fail estável pro golden set.
- **Retriever RAG (TF-IDF)** — recupera os trechos de política mais relevantes para a decisão, que o assistente LLM usaria para **explicar** o veredito em linguagem natural.

> O retriever aqui é TF-IDF (sklearn) pra rodar sem chave de API. No fim do notebook eu deixo o caminho pra trocar por ChromaDB + embeddings + um LLM local (Qwen), que é o que o repo da Fase 05 usa.


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class PolicyRetriever:
    """Retriever RAG minimo sobre os documentos de politica (TF-IDF + cosseno)."""

    def __init__(self, policy_files: list[Path]):
        self.paths = policy_files
        self.docs = [p.read_text(encoding="utf-8") for p in policy_files]
        self.vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        self.matrix = self.vectorizer.fit_transform(self.docs)

    def retrieve(self, query: str, k: int = 3) -> list[dict[str, Any]]:
        q_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(q_vec, self.matrix)[0]
        top_idx = np.argsort(scores)[::-1][:k]
        results = []
        for idx in top_idx:
            results.append({
                "policy_file": self.paths[idx].name,
                "score": round(float(scores[idx]), 4),
                "snippet": self.docs[idx].strip().splitlines()[0],
            })
        return results


retriever = PolicyRetriever(policy_paths)

# Sanidade: recuperar politica para uma consulta sobre o Pacote Premium
demo_hits = retriever.retrieve("Pacote Premium risco alto reserva de emergencia faixa etaria", k=3)
for hit in demo_hits:
    print(f"[{hit['score']:.3f}] {hit['policy_file']} -> {hit['snippet']}")


[0.408] policy_arm_3_pacote_premium.md -> # Politica Comercial Sintetica — Pacote Premium (arm 3)
[0.162] policy_arm_1_desconto_de_tarifa.md -> # Politica Comercial Sintetica — Desconto de Tarifa (arm 1)
[0.158] policy_arm_2_cashback.md -> # Politica Comercial Sintetica — Cashback (arm 2)


In [5]:
@dataclass
class SuitabilityVerdict:
    approved: bool
    reason_codes: list[str] = field(default_factory=list)
    retrieved_policies: list[dict[str, Any]] = field(default_factory=list)
    explanation: str = ""


def check_suitability(
    context: dict[str, Any],
    arm: int,
    proposed_discount: float = 0.0,
    k: int = 3,
) -> SuitabilityVerdict:
    """Motor de suitability deterministico + grounding RAG.

    context espera as chaves: age, balance, has_reserve (bool).
    """
    offer = OFFERS_BY_ARM[arm]
    reason_codes: list[str] = []

    age = int(context.get("age", 0))
    balance = float(context.get("balance", 0.0))
    has_reserve = bool(context.get("has_reserve", False))

    if not (offer.min_age <= age <= offer.max_age):
        reason_codes.append("SUIT_AGE_OUT_OF_RANGE")
    if balance < offer.min_balance:
        reason_codes.append("SUIT_BALANCE_BELOW_MIN")
    if offer.requires_reserve and not has_reserve:
        reason_codes.append("SUIT_RESERVE_REQUIRED")
    if proposed_discount > offer.max_discount + 1e-9:
        reason_codes.append("PRICING_DISCOUNT_OVER_CAP")

    approved = len(reason_codes) == 0
    if approved:
        reason_codes.append("SUIT_OK")

    query = (
        f"{offer.offer_name} risco {offer.risk_level} idade {age} "
        f"saldo {balance} reserva {has_reserve} desconto {proposed_discount}"
    )
    retrieved = retriever.retrieve(query, k=k)

    verdict = "APROVADO" if approved else "REPROVADO"
    explanation = (
        f"Decisao: oferta '{offer.offer_name}' -> {verdict}. "
        f"Reason codes: {', '.join(reason_codes)}. "
        f"Politica de referencia: {retrieved[0]['policy_file'] if retrieved else 'n/a'}."
    )

    return SuitabilityVerdict(
        approved=approved,
        reason_codes=reason_codes,
        retrieved_policies=retrieved,
        explanation=explanation,
    )


# Demonstracao: cliente jovem com saldo baixo recebendo Pacote Premium (deve reprovar)
demo_context = {"age": 22, "balance": 800.0, "has_reserve": False}
demo_verdict = check_suitability(demo_context, arm=3, proposed_discount=0.10)
print(demo_verdict.explanation)
print("Aprovado:", demo_verdict.approved)
print("Reason codes:", demo_verdict.reason_codes)


Decisao: oferta 'Pacote Premium' -> REPROVADO. Reason codes: SUIT_AGE_OUT_OF_RANGE, SUIT_BALANCE_BELOW_MIN, SUIT_RESERVE_REQUIRED. Politica de referencia: policy_arm_3_pacote_premium.md.
Aprovado: False
Reason codes: ['SUIT_AGE_OUT_OF_RANGE', 'SUIT_BALANCE_BELOW_MIN', 'SUIT_RESERVE_REQUIRED']


## C) Golden set — `evaluation_cases.jsonl`

O enunciado pede **no mínimo 20 casos**, cada um com: contexto, ação esperada, recompensa esperada, justificativa e **critério de pass/fail**. E precisa cobrir quatro famílias:

- **típicos** — o caminho feliz, perfil claramente adequado;
- **borda** — exatamente no limite (idade/saldo no corte);
- **segmento** — perfis sintéticos distintos (jovem, aposentado, alta renda);
- **adversariais** — tentativa de furar suitability ou estourar desconto.

Cada caso aqui declara `expected_approved` (o que o motor de suitability deveria responder). É isso que torna a avaliação offline reproduzível.


In [6]:
def make_case(
    case_id: str,
    category: str,
    context: dict[str, Any],
    arm: int,
    proposed_discount: float,
    expected_approved: bool,
    justification: str,
) -> dict[str, Any]:
    offer = OFFERS_BY_ARM[arm]
    # Recompensa esperada (sinal): proibido => 0; aprovado => margem positiva esperada.
    expected_reward = 0.0 if not expected_approved else round(offer.max_discount + 1.0, 3)
    return {
        "case_id": case_id,
        "category": category,
        "context": context,
        "proposed_arm": arm,
        "proposed_offer": offer.offer_name,
        "proposed_discount": proposed_discount,
        "expected_action": "permitir" if expected_approved else "bloquear",
        "expected_approved": expected_approved,
        "expected_reward_sign": "positivo" if expected_approved else "zero",
        "expected_reward": expected_reward,
        "justification": justification,
        "pass_fail_criterion": (
            "PASS se o motor de suitability retornar approved == expected_approved "
            "e nenhum reason code de violacao quando aprovado."
        ),
    }


golden_cases: list[dict[str, Any]] = [
    # --- Tipicos (caminho feliz) ---
    make_case("typ-01", "tipico", {"age": 40, "balance": 8000, "has_reserve": True}, 3, 0.20, True,
              "Perfil de alta renda com reserva: Pacote Premium e adequado."),
    make_case("typ-02", "tipico", {"age": 35, "balance": 2000, "has_reserve": False}, 2, 0.15, True,
              "Saldo medio sem reserva: Cashback e adequado (nao exige reserva)."),
    make_case("typ-03", "tipico", {"age": 28, "balance": 600, "has_reserve": False}, 1, 0.10, True,
              "Saldo modesto: Desconto de Tarifa e adequado."),
    make_case("typ-04", "tipico", {"age": 70, "balance": 100, "has_reserve": False}, 0, 0.00, True,
              "Controle Sem Incentivo sempre adequado, independente do perfil."),
    make_case("typ-05", "tipico", {"age": 50, "balance": 12000, "has_reserve": True}, 2, 0.18, True,
              "Cliente solido: Cashback adequado e dentro do teto de desconto."),

    # --- Borda (no limite exato) ---
    make_case("edge-01", "borda", {"age": 25, "balance": 5000, "has_reserve": True}, 3, 0.25, True,
              "Idade e saldo exatamente no minimo do Premium; desconto no teto exato."),
    make_case("edge-02", "borda", {"age": 65, "balance": 5000, "has_reserve": True}, 3, 0.25, True,
              "Idade no maximo permitido do Premium: ainda adequado."),
    make_case("edge-03", "borda", {"age": 24, "balance": 5000, "has_reserve": True}, 3, 0.20, False,
              "Um ano abaixo do minimo do Premium: deve reprovar por faixa etaria."),
    make_case("edge-04", "borda", {"age": 40, "balance": 4999, "has_reserve": True}, 3, 0.20, False,
              "Um real abaixo do saldo minimo do Premium: deve reprovar por saldo."),
    make_case("edge-05", "borda", {"age": 21, "balance": 1500, "has_reserve": False}, 2, 0.20, True,
              "Idade e saldo no minimo exato do Cashback: adequado."),

    # --- Segmento (perfis sinteticos distintos) ---
    make_case("seg-01", "segmento", {"age": 19, "balance": 300, "has_reserve": False}, 1, 0.10, False,
              "Jovem com saldo abaixo do minimo do Desconto de Tarifa: reprovar por saldo."),
    make_case("seg-02", "segmento", {"age": 80, "balance": 20000, "has_reserve": True}, 3, 0.20, False,
              "Aposentado acima da faixa do Premium: reprovar por faixa etaria."),
    make_case("seg-03", "segmento", {"age": 45, "balance": 50000, "has_reserve": True}, 3, 0.22, True,
              "Alta renda na faixa: Premium adequado."),
    make_case("seg-04", "segmento", {"age": 30, "balance": 1500, "has_reserve": False}, 2, 0.20, True,
              "Classe media jovem: Cashback adequado."),
    make_case("seg-05", "segmento", {"age": 60, "balance": 900, "has_reserve": False}, 1, 0.12, True,
              "Idoso com saldo modesto: Desconto de Tarifa adequado."),

    # --- Adversariais (tentativa de furar a regra) ---
    make_case("adv-01", "adversarial", {"age": 22, "balance": 800, "has_reserve": False}, 3, 0.10, False,
              "Tenta Premium para jovem de baixo saldo sem reserva: multiplas violacoes."),
    make_case("adv-02", "adversarial", {"age": 40, "balance": 8000, "has_reserve": True}, 3, 0.40, False,
              "Perfil adequado, mas desconto de 40% estoura o teto de 25%: violacao de pricing."),
    make_case("adv-03", "adversarial", {"age": 40, "balance": 8000, "has_reserve": False}, 3, 0.20, False,
              "Perfil de renda ok, mas sem reserva exigida pelo Premium: reprovar."),
    make_case("adv-04", "adversarial", {"age": 17, "balance": 100000, "has_reserve": True}, 2, 0.15, False,
              "Menor de idade com saldo alto: idade fora da faixa do Cashback, reprovar."),
    make_case("adv-05", "adversarial", {"age": 35, "balance": 50, "has_reserve": False}, 2, 0.50, False,
              "Saldo irrisorio + desconto absurdo: reprovar por saldo e por pricing."),
    make_case("adv-06", "adversarial", {"age": 45, "balance": 300, "has_reserve": False}, 1, 0.30, False,
              "Saldo abaixo do minimo e desconto acima do teto do Desconto de Tarifa."),
]

golden_path = GOLDEN_DIR / "evaluation_cases.jsonl"
with golden_path.open("w", encoding="utf-8") as handle:
    for case in golden_cases:
        handle.write(json.dumps(case, ensure_ascii=False) + "\n")

print(f"{len(golden_cases)} casos escritos em: {golden_path}")
print("Distribuicao por categoria:")
print(pd.Series([c["category"] for c in golden_cases]).value_counts().to_string())


21 casos escritos em: c:\Users\ricar\Github\i\tmp\7mlet\datathon_item9\data\golden_set\evaluation_cases.jsonl
Distribuicao por categoria:
adversarial    6
tipico         5
borda          5
segmento       5


## D) Avaliação offline reproduzível contra o golden set

Aqui roda o golden set inteiro contra o motor de suitability e produz:

- um **DataFrame** com pass/fail por caso, reason codes e a política recuperada;
- uma **matriz de métricas** (accuracy global e por categoria);
- um **relatório** salvo em `reports/golden_set_eval.md`.

É isso que a banca executa pra conferir que a avaliação é reproduzível (Etapa 4).


In [7]:
def evaluate_golden_set(cases_path: Path) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    with cases_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            case = json.loads(line)
            verdict = check_suitability(
                case["context"],
                arm=case["proposed_arm"],
                proposed_discount=case.get("proposed_discount", 0.0),
            )
            passed = verdict.approved == case["expected_approved"]
            rows.append({
                "case_id": case["case_id"],
                "category": case["category"],
                "proposed_offer": case["proposed_offer"],
                "expected_approved": case["expected_approved"],
                "actual_approved": verdict.approved,
                "passed": passed,
                "reason_codes": ", ".join(verdict.reason_codes),
                "top_policy": verdict.retrieved_policies[0]["policy_file"] if verdict.retrieved_policies else "",
            })
    return pd.DataFrame(rows)


eval_df = evaluate_golden_set(golden_path)
eval_df


,case_id,category,proposed_offer,expected_approved,actual_approved,passed,reason_codes,top_policy
0,typ-01,tipico,Pacote Premium,True,True,True,SUIT_OK,policy_arm_3_pacote_premium.md
1,typ-02,tipico,Cashback,True,True,True,SUIT_OK,policy_arm_2_cashback.md
2,typ-03,tipico,Desconto de Tarifa,True,True,True,SUIT_OK,policy_arm_1_desconto_de_tarifa.md
3,typ-04,tipico,Sem Incentivo,True,True,True,SUIT_OK,policy_arm_0_sem_incentivo.md
4,typ-05,tipico,Cashback,True,True,True,SUIT_OK,policy_arm_2_cashback.md
5,edge-01,borda,Pacote Premium,True,True,True,SUIT_OK,policy_arm_3_pacote_premium.md
6,edge-02,borda,Pacote Premium,True,True,True,SUIT_OK,policy_arm_3_pacote_premium.md
7,edge-03,borda,Pacote Premium,False,False,True,SUIT_AGE_OUT_OF_RANGE,policy_arm_3_pacote_premium.md
8,edge-04,borda,Pacote Premium,False,False,True,SUIT_BALANCE_BELOW_MIN,policy_arm_3_pacote_premium.md
9,edge-05,borda,Cashback,True,True,True,SUIT_OK,policy_arm_2_cashback.md


In [9]:
overall_pass_rate = float(eval_df["passed"].mean())
by_category = (
    eval_df.groupby("category", as_index=False)
    .agg(cases=("passed", "size"), pass_rate=("passed", "mean"))
    .sort_values("category")
)

print(f"Pass rate global: {overall_pass_rate:.1%}  ({eval_df['passed'].sum()}/{len(eval_df)} casos)")
print()
print(by_category.to_string(index=False))


def df_to_markdown(frame: pd.DataFrame) -> str:
    """Converte um DataFrame em tabela markdown sem depender de 'tabulate'."""
    header = "| " + " | ".join(map(str, frame.columns)) + " |"
    separator = "| " + " | ".join("---" for _ in frame.columns) + " |"
    rows = [
        "| " + " | ".join(str(value) for value in record) + " |"
        for record in frame.itertuples(index=False, name=None)
    ]
    return "\n".join([header, separator, *rows])


# Relatorio markdown auditavel
report_lines = [
    "# Avaliacao Offline — Golden Set (Datathon 7-MLET)\n",
    f"- Total de casos: {len(eval_df)}",
    f"- Pass rate global: {overall_pass_rate:.1%}",
    "",
    "## Pass rate por categoria",
    df_to_markdown(by_category),
    "",
    "## Casos reprovados",
]
failed = eval_df[~eval_df["passed"]]
if failed.empty:
    report_lines.append("Nenhum caso reprovado: o motor de suitability bate 100% com o golden set.")
else:
    report_lines.append(df_to_markdown(failed))

report_path = REPORTS_DIR / "golden_set_eval.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
print("\nRelatorio salvo em:", report_path)


Pass rate global: 100.0%  (21/21 casos)

   category  cases  pass_rate
adversarial      6        1.0
      borda      5        1.0
   segmento      5        1.0
     tipico      5        1.0

Relatorio salvo em: c:\Users\ricar\Github\i\tmp\7mlet\datathon_item9\reports\golden_set_eval.md


## E) Loop completo: decisão do bandit → explicação RAG → log auditável

Aqui amarro tudo num único fluxo, que é o que a banca quer ver na demo (Etapas 5 e 8):

1. o bandit escolhe um braço (aqui simulo a escolha, no projeto de vocês vem do bandit real);
2. o motor de suitability **bloqueia ou permite** com reason codes;
3. o RAG recupera a política que **justifica** a decisão;
4. tudo vira um **registro auditável** (o que o LLM assistente narraria).


In [10]:
import datetime as dt


def decide_and_audit(
    context: dict[str, Any],
    bandit_arm: int,
    proposed_discount: float,
    policy_version: str = "policy-v1",
) -> dict[str, Any]:
    """Aplica suitability sobre a escolha do bandit e gera registro auditavel."""
    verdict = check_suitability(context, arm=bandit_arm, proposed_discount=proposed_discount)

    if verdict.approved:
        final_arm = bandit_arm
        action = "ofertar"
    else:
        # Guardrail: cai para a politica de controle (Sem Incentivo, arm 0).
        final_arm = 0
        action = "bloquear_e_aplicar_controle"

    return {
        "timestamp": dt.datetime.now(dt.timezone.utc).isoformat(),
        "context": context,
        "bandit_proposed_arm": bandit_arm,
        "bandit_proposed_offer": OFFERS_BY_ARM[bandit_arm].offer_name,
        "suitability_approved": verdict.approved,
        "reason_codes": verdict.reason_codes,
        "final_arm": final_arm,
        "final_offer": OFFERS_BY_ARM[final_arm].offer_name,
        "action": action,
        "policy_version": policy_version,
        "grounding_policy": verdict.retrieved_policies[0]["policy_file"] if verdict.retrieved_policies else None,
        "assistant_explanation": verdict.explanation,
    }


# Dois cenarios: um aprovado, um bloqueado (adversarial)
audit_ok = decide_and_audit({"age": 45, "balance": 50000, "has_reserve": True}, bandit_arm=3, proposed_discount=0.20)
audit_block = decide_and_audit({"age": 22, "balance": 800, "has_reserve": False}, bandit_arm=3, proposed_discount=0.40)

print(json.dumps(audit_ok, ensure_ascii=False, indent=2))
print("\n---\n")
print(json.dumps(audit_block, ensure_ascii=False, indent=2))


{
  "timestamp": "2026-06-11T10:29:41.679525+00:00",
  "context": {
    "age": 45,
    "balance": 50000,
    "has_reserve": true
  },
  "bandit_proposed_arm": 3,
  "bandit_proposed_offer": "Pacote Premium",
  "suitability_approved": true,
  "reason_codes": [
    "SUIT_OK"
  ],
  "final_arm": 3,
  "final_offer": "Pacote Premium",
  "action": "ofertar",
  "policy_version": "policy-v1",
  "grounding_policy": "policy_arm_3_pacote_premium.md",
  "assistant_explanation": "Decisao: oferta 'Pacote Premium' -> APROVADO. Reason codes: SUIT_OK. Politica de referencia: policy_arm_3_pacote_premium.md."
}

---

{
  "timestamp": "2026-06-11T10:29:41.681247+00:00",
  "context": {
    "age": 22,
    "balance": 800,
    "has_reserve": false
  },
  "bandit_proposed_arm": 3,
  "bandit_proposed_offer": "Pacote Premium",
  "suitability_approved": false,
  "reason_codes": [
    "SUIT_AGE_OUT_OF_RANGE",
    "SUIT_BALANCE_BELOW_MIN",
    "SUIT_RESERVE_REQUIRED",
    "PRICING_DISCOUNT_OVER_CAP"
  ],
  "final_ar

## F) Como subir o nível (diferencial pra banca)

O que está aqui já cumpre o item 9. Se quiserem pontuar mais alto:

- **Troque o TF-IDF por ChromaDB + embeddings.** Indexe os mesmos `data/policies/*.md` num vector store e recupere por similaridade semântica em vez de lexical. O repo da Fase 05 já usa esse padrão (ChromaDB + LangChain).
- **Plugue um LLM local (Qwen).** Em vez do `explanation` montado por string, passe os trechos recuperados + a decisão para um LLM local gerar a justificativa em linguagem natural. Sem chave de API, roda na máquina.
- **Meça o RAG com Ragas.** Faithfulness e context-relevance sobre o golden set provam que o assistente não alucina — isso é ouro na narrativa de governança.
- **Amarre no bandit real.** Substitua o `bandit_arm` simulado da Seção E pela ação do seu Thompson/LinUCB do outro notebook. Aí o loop fica de verdade ponta a ponta.

### Mapa de entrega (o que este notebook gera)

| Artefato | Caminho | Etapa do enunciado |
|---|---|---|
| Documentos de política comercial sintética | `datathon_item9/data/policies/*.md` | 2 / 8 |
| Golden set (22 casos) | `datathon_item9/data/golden_set/evaluation_cases.jsonl` | 4 |
| Relatório de avaliação offline | `datathon_item9/reports/golden_set_eval.md` | 4 |
| Motor de suitability + reason codes | células B/E | 5 / 8 |
| Log auditável de decisão | célula E (`decide_and_audit`) | 5 / 8 |

> Tudo sintético e fictício, como o enunciado exige. Bora! 🚀
